In [0]:
%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, row_number
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@adlsg2rag.dfs.core.windows.net/sqlserver/customers/load_date=2026-03-13/"

In [0]:
df = spark.read.format("parquet").load(bronze_path)

# Infer schema happens automatically here through spark.read.parquet
df.printSchema()

In [0]:
# Null / zero PK check
df_clean = df.filter(col("CustomerID").isNotNull() & (col("CustomerID") != 0))

In [0]:
# Adding Timestamp column
df_clean = df_clean.withColumn("processed_ts", current_timestamp())

In [0]:
# Duplicate removal
w = Window.partitionBy("CustomerID").orderBy(col("processed_ts").desc())
df_clean = df_clean.withColumn("rn", row_number().over(w)) \
                   .filter(col("rn") == 1) \
                   .drop("rn")

In [0]:
# Remove unnecessary columns / rename columns
df_clean = df_clean.select(
    col("CustomerID").alias("src_CustomerID"),
    col("CustomerName").alias("src_CustomerName"),
    col("City").alias("src_City"),
    col("Country").alias("src_Country"),
    col("processed_ts")
)

In [0]:


spark.sql("""
CREATE TABLE IF NOT EXISTS adbrag.silver.customers_silver
(
  src_CustomerID INT,
  src_CustomerName STRING,
  src_City STRING,
  src_Country STRING,
  processed_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_CustomerID)
""")



In [0]:
# Write to managed Delta table
df_clean.write.mode("overwrite").format("delta").saveAsTable("adbrag.silver.customers_silver")